In [2]:
import pandas as pd
import numpy as np

import os

def load_hotel_reserve():
  customer_tb = pd.read_csv('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/customer.csv')
  hotel_tb = pd.read_csv('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/hotel.csv')
  reserve_tb = pd.read_csv('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/reserve.csv')
  return customer_tb, hotel_tb, reserve_tb


def load_holiday_mst():
  holiday_tb = pd.read_csv('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/holiday_mst.csv',
                           index_col=False)
  return holiday_tb


def load_production():
  production_tb = pd.read_csv('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/production.csv')
  return production_tb


def load_production_missing_num():
  production_tb = pd.read_csv('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/production_missing_num.csv')
  return production_tb


def load_production_missing_category():
  production_tb = pd.read_csv('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/production_missing_category.csv')
  return production_tb


def load_monthly_index():
  monthly_index_tb = \
    pd.read_csv('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/monthly_index.csv')
  return monthly_index_tb


def load_meros_txt():
  with open('/workspaces/Preprocessing-excersize/notebooks/Preprocessing-excersize/data/txt/meros.txt', 'r') as f:
    meros = f.read()
    f.close()
  return meros


In [3]:
customer_tb, hotel_tb, reserve_tb = load_hotel_reserve()

In [4]:
df= pd.DataFrame({'value': [40000/3]})

print(df.dtypes)
print('-'*15)

print(df['value'].astype('int8'))
print(df['value'].astype('int16'))
print(df['value'].astype('int32'))
print(df['value'].astype('int64'))
print('-'*15)

print(df['value'].astype('float16'))
print(df['value'].astype('float32'))
print(df['value'].astype('float64'))
print(df['value'].astype('float128'))
print('-'*15)

print(df['value'].astype(int))
print(df['value'].astype(float))

value    float64
dtype: object
---------------
0    21
Name: value, dtype: int8
0    13333
Name: value, dtype: int16
0    13333
Name: value, dtype: int32
0    13333
Name: value, dtype: int64
---------------
0    13336.0
Name: value, dtype: float16
0    13333.333008
Name: value, dtype: float32
0    13333.333333
Name: value, dtype: float64
0    13333.333333
Name: value, dtype: float128
---------------
0    13333
Name: value, dtype: int64
0    13333.333333
Name: value, dtype: float64


/usr/local/python/3.12.1/lib/python3.12/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [ ]:
# 가격 데이터를 로그로 변환해서 데이터 분포를 균일하게 만드는 것
# 가격 데이터의 값 차이가 너무 크면 머신러닝 모델이 학습을 제대로 못해 그래서 로그를 씌워서 차이를 줄이는 것
#원래 가격이 로그로 압축된 결과 : 숫자가 1이면 만원대, 2면 십만원대, 3이면 백만원대
reserve_tb['total_price_log']=\
    reserve_tb['total_price'].apply(lambda x: np.log10( x / 1000 +1)) #원 단위를 천원 단위로 줄이기

reserve_tb['total_price_log']

0       1.992111
1       1.334454
2       1.539076
3       2.290925
4       1.839478
          ...   
4025    1.230449
4026    1.631444
4027    1.879669
4028    2.733197
4029    1.654177
Name: total_price_log, Length: 4030, dtype: float64

In [ ]:
#나이를 10대,20대,30대로 묶는 코드
customer_tb['age_rank']= \
    (np.floor(customer_tb['age'] / 10) * 10).astype('category')
    # /10은 10으로 나누고 27살이 2.7 그리고 np.floor로 소수점 버리고 내리기. 그러면 2.0 여기에 다시 10 곱하기
    # 그러면 20, 30이 됨. 그리고 이걸 숫자가 아니라 그룹으로 취급
    
print(customer_tb['age_rank'])

0      40.0
1      30.0
2      40.0
3      40.0
4      30.0
       ... 
995    40.0
996    30.0
997    30.0
998    40.0
999    30.0
Name: age_rank, Length: 1000, dtype: category
Categories (7, float64): [20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0]


## 정규화가 필요한 이유

머신러닝 모델은 숫자 크기에 영향을 받음

| 컬럼 | 범위 | 문제 |
|------|------|------|
| people_num | 1 ~ 5 | 너무 작아서 무시됨 |
| total_price | 10,000 ~ 500,000 | 너무 커서 학습을 지배함 |

→ StandardScaler로 평균=0, 표준편차=1 기준으로 맞춰서 해결

In [6]:
# 서로 단위가 다른 데이터를 같은 기준으로 맞추는 것(정규화)
from sklearn.preprocessing import StandardScaler    #StandatScaler는 평균을 0,표준편차를 1로 맞춰줌

reserve_tb['people_num']= reserve_tb['people_num'].astype(float)    #float타입만 받아서 변환

ss = StandardScaler()       # 정규화 객체 생성

result = ss.fit_transform(reserve_tb[['people_num', 'total_price']])
# fit은 평균이랑 표준편차를 계산. transform은 그걸로 변환. 둘을 한번에 하는게 fit_transform

reserve_tb['people_num_normalized'] = [x[0] for x in result]    #각 행의 첫번째 값을 컬럼으로 분리
reserve_tb['total_price_normalized'] = [x[1] for x in result]   #각 행의 두번째 값을 컬럼으로 분리

reserve_tb

,reserve_id,hotel_id,customer_id,reserve_datetime,checkin_date,checkin_time,checkout_date,people_num,total_price,total_price_log,people_num_normalized,total_price_normalized
0,r1,h_75,c_1,2016-03-06 13:09:42,2016-03-26,10:00:00,2016-03-29,4.0,97200,1.992111,1.300709,-0.053194
1,r2,h_219,c_1,2016-07-16 23:39:55,2016-07-20,11:30:00,2016-07-21,2.0,20600,1.334454,-0.483753,-0.747822
2,r3,h_179,c_1,2016-09-24 10:03:17,2016-10-19,09:00:00,2016-10-22,2.0,33600,1.539076,-0.483753,-0.629935
3,r4,h_214,c_1,2017-03-08 03:20:10,2017-03-29,11:00:00,2017-03-30,4.0,194400,2.290925,1.300709,0.828240
4,r5,h_16,c_1,2017-09-05 19:50:37,2017-09-22,10:30:00,2017-09-23,3.0,68100,1.839478,0.408478,-0.317080
...,...,...,...,...,...,...,...,...,...,...,...,...
4025,r4026,h_129,c_999,2017-06-27 23:00:02,2017-07-10,09:30:00,2017-07-11,2.0,16000,1.230449,-0.483753,-0.789536
4026,r4027,h_97,c_999,2017-09-29 05:24:57,2017-10-09,10:30:00,2017-10-10,2.0,41800,1.631444,-0.483753,-0.555575
4027,r4028,h_27,c_999,2018-03-14 05:01:45,2018-04-02,11:30:00,2018-04-04,2.0,74800,1.879669,-0.483753,-0.256323
4028,r4029,h_48,c_1000,2016-04-16 15:20:17,2016-05-10,09:30:00,2016-05-13,4.0,540000,2.733197,1.300709,3.962229
